In [4]:
# === gridsearch_tuning.ipynb — Setup ===
import os
import glob
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Load the FROZEN dataset from the central data/model_dataset/ folder.
# Fall back to a recursive search in case the folder structure ever changes.
fixed_path = '../../data/model_dataset/model_dataset.csv'
matches = [fixed_path] if os.path.exists(fixed_path) else glob.glob('../../**/model_dataset.csv', recursive=True)
if not matches:
    raise FileNotFoundError("model_dataset.csv not found — run baseline_comparison first")
print("Loading:", matches[0])
model_df = pd.read_csv(matches[0])

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']

# Spatial blocks
BLOCK = 0.25
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                    (model_df['lat']//BLOCK).astype(int).astype(str)

print("Dataset:", model_df.shape, "| blocks:", model_df['block'].nunique())

Loading: ../../data/model_dataset/model_dataset.csv
Dataset: (6231, 14) | blocks: 120


In [6]:
# === GridSearchCV — hyperparameter tuning with 10-fold spatial-block CV ===
# CV: GroupKFold(n_splits=10). groups = spatial block id (0.25 deg ~ 27 km cells),
# so a given block's rows NEVER get split across train/validation within a fold
# -> no spatial data leakage. Every fit trains on 9/10 folds and validates on the
# 1 held-out fold; GridSearchCV repeats this for EVERY hyperparameter candidate.
X = model_df[pred_cols].values
y = model_df['burned'].astype(int).values
groups = model_df['block'].values
cv = GroupKFold(n_splits=10)

# ---------- Logistic Regression grid ----------
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000)),
])
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear'],   # only solver here that supports both l1 and l2
}
# 4 x 2 x 1 = 8 candidates x 10 folds = 80 fits (9 folds train / 1 fold validate
# per fit). Scored with PR-AUC (scoring='average_precision'), averaged across the
# 10 folds per candidate; the candidate with the best mean PR-AUC wins.
lr_search = GridSearchCV(pipeline_lr, param_grid_lr, cv=cv,
                          scoring='average_precision', n_jobs=-1)
lr_search.fit(X, y, groups=groups)   # groups passed at .fit() time, NOT inside cv=...
lr_search_best_parameters = lr_search.best_params_

# ---------- Random Forest grid ----------
param_grid_rf = {
    'n_estimators':     [200, 300, 500],
    'max_features':     ['sqrt', 0.5],
    'min_samples_leaf': [3, 5, 10, 20],
    'max_depth':        [None, 10, 20],
}
# 3 x 2 x 4 x 3 = 72 candidates x 10 folds = 720 fits (9 folds train / 1 fold
# validate per fit). Each fit is scored with PR-AUC (scoring='average_precision');
# GridSearchCV averages the 10 fold scores per candidate and keeps the candidate
# with the best mean PR-AUC.
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=cv,
    scoring='average_precision',
    n_jobs=-1,
)
rf_search.fit(X, y, groups=groups)
rf_search_best_parameters = rf_search.best_params_

n_candidates_lr = (len(param_grid_lr['clf__C']) * len(param_grid_lr['clf__penalty'])
                   * len(param_grid_lr['clf__solver']))
n_candidates_rf = (len(param_grid_rf['n_estimators']) * len(param_grid_rf['max_features'])
                   * len(param_grid_rf['min_samples_leaf']) * len(param_grid_rf['max_depth']))
print(f"LR grid: {n_candidates_lr} candidates x 10 folds = {n_candidates_lr * 10} fits")
print(f"RF grid: {n_candidates_rf} candidates x 10 folds = {n_candidates_rf * 10} fits")
print("Best LR hyperparameters:", lr_search_best_parameters)
print("Best RF hyperparameters:", rf_search_best_parameters)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LR grid: 8 candidates x 10 folds = 80 fits
RF grid: 72 candidates x 10 folds = 720 fits
Best LR hyperparameters: {'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Best RF hyperparameters: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 500}


In [7]:
# === Tuned models — full evaluation with the SAME output format as before ===
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.base import clone

# The tuned models (best configs found by GridSearch)
best_lr = lr_search.best_estimator_      # Pipeline: StandardScaler + LogisticRegression(C=0.01, L1)
best_rf = rf_search.best_estimator_      # RandomForest(n_est=500, sqrt, leaf=5, depth=None)

print("Tuned LR:", lr_search.best_params_)
print("Tuned RF:", rf_search.best_params_)

Tuned LR: {'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Tuned RF: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 500}


In [8]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=0.25):
    """Spatial block CV (with uncertainty) + temporal split — identical format for any model."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values

    # Spatial blocks
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    # ---------- SPATIAL BLOCK CV ----------
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        m = clone(estimator).fit(X[tr], y[tr])          # fresh copy, refit per fold
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc   = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1    = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        print(f"  Fold {fold:2d}: AUC={auc:.3f}  PR-AUC={prauc:.3f}  F1={f1:.3f}")

    r = np.array(rows)
    print(f"\nSPATIAL BLOCK CV — {name}")
    print(f"  AUC-ROC : {r[:,0].mean():.3f} ± {r[:,0].std():.3f}")
    print(f"  PR-AUC  : {r[:,1].mean():.3f} ± {r[:,1].std():.3f}")
    print(f"  F1      : {r[:,2].mean():.3f} ± {r[:,2].std():.3f}")

    # ---------- TEMPORAL SPLIT ----------
    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])

    print(f"\nTEMPORAL SPLIT — {name}")
    print(f"  train ≤2019: {tr.sum()} rows ({int(y[tr].sum())} events) | "
          f"test ≥2020: {te.sum()} rows ({int(y[te].sum())} events)")
    print(f"  AUC-ROC : {roc_auc_score(y[te], prob):.3f}")
    print(f"  PR-AUC  : {average_precision_score(y[te], prob):.3f}")
    print(f"  F1      : {f1_score(y[te], pred):.3f}")

    return {'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0),
            'temporal': (roc_auc_score(y[te], prob),
                         average_precision_score(y[te], prob),
                         f1_score(y[te], pred))}

In [9]:
# evaluate the tuned models

res_lr = evaluate_model("Logistic Regression (tuned)", best_lr, model_df, pred_cols)


res_rf = evaluate_model("Random Forest (tuned)", best_rf, model_df, pred_cols)

c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  Fold  1: AUC=0.896  PR-AUC=0.833  F1=0.803
  Fold  2: AUC=0.684  PR-AUC=0.517  F1=0.460
  Fold  3: AUC=0.850  PR-AUC=0.762  F1=0.701
  Fold  4: AUC=0.853  PR-AUC=0.788  F1=0.670
  Fold  5: AUC=0.881  PR-AUC=0.717  F1=0.502
  Fold  6: AUC=0.847  PR-AUC=0.716  F1=0.493
  Fold  7: AUC=0.834  PR-AUC=0.658  F1=0.637
  Fold  8: AUC=0.867  PR-AUC=0.799  F1=0.687
  Fold  9: AUC=0.769  PR-AUC=0.533  F1=0.441
  Fold 10: AUC=0.872  PR-AUC=0.726  F1=0.690

SPATIAL BLOCK CV — Logistic Regression (tuned)
  AUC-ROC : 0.835 ± 0.060
  PR-AUC  : 0.705 ± 0.102
  F1      : 0.608 ± 0.118

TEMPORAL SPLIT — Logistic Regression (tuned)
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.807
  PR-AUC  : 0.618
  F1      : 0.551


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  Fold  1: AUC=0.915  PR-AUC=0.878  F1=0.823
  Fold  2: AUC=0.776  PR-AUC=0.647  F1=0.618
  Fold  3: AUC=0.867  PR-AUC=0.779  F1=0.708
  Fold  4: AUC=0.883  PR-AUC=0.818  F1=0.739
  Fold  5: AUC=0.909  PR-AUC=0.768  F1=0.585
  Fold  6: AUC=0.887  PR-AUC=0.816  F1=0.667
  Fold  7: AUC=0.898  PR-AUC=0.775  F1=0.658
  Fold  8: AUC=0.881  PR-AUC=0.812  F1=0.696
  Fold  9: AUC=0.843  PR-AUC=0.693  F1=0.589
  Fold 10: AUC=0.916  PR-AUC=0.845  F1=0.755

SPATIAL BLOCK CV — Random Forest (tuned)
  AUC-ROC : 0.877 ± 0.040
  PR-AUC  : 0.783 ± 0.066
  F1      : 0.684 ± 0.072

TEMPORAL SPLIT — Random Forest (tuned)
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.816
  PR-AUC  : 0.558
  F1      : 0.561


In [11]:
# --- LR coefficients (from the tuned pipeline) ---
coefs = best_lr.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'predictor': pred_cols, 'coefficient': coefs}) \
            .sort_values('coefficient', key=abs, ascending=False)
lr_c = best_lr.named_steps['clf'].C
print(f"\nStandardized coefficients — TUNED LR (L1, C={lr_c}):")
print(coef_df.round(3).to_string(index=False))
zeroed = coef_df[coef_df['coefficient'] == 0]['predictor'].tolist()
print(f"Eliminated by L1: {zeroed if zeroed else 'none'}")

# --- RF impurity importance ---
X_all = model_df[pred_cols].values
y_all = model_df['burned'].astype(int).values
rf_fit = clone(best_rf).fit(X_all, y_all)
imp = pd.DataFrame({'predictor': pred_cols, 'importance': rf_fit.feature_importances_}) \
        .sort_values('importance', ascending=False)
print("\nRF impurity importance — TUNED:")
print(imp.round(3).to_string(index=False))


Standardized coefficients — TUNED LR (L1, C=0.1):
  predictor  coefficient
       ndvi       -0.763
    wind_ms        0.731
    vpd_kPa        0.489
 dist_parks        0.361
     temp_C        0.335
 dist_roads       -0.193
dist_mosaic       -0.110
  dist_coca        0.013
        oni        0.000
Eliminated by L1: ['oni']

RF impurity importance — TUNED:
  predictor  importance
       ndvi       0.240
    wind_ms       0.184
    vpd_kPa       0.149
     temp_C       0.097
 dist_parks       0.090
        oni       0.081
 dist_roads       0.069
  dist_coca       0.056
dist_mosaic       0.034


In [12]:
import pandas as pd

# Results already computed (spatial CV: mean ± std | temporal: single value)
results = {
    'LR — before tuning': {
        'auc_sp': (0.836, 0.029), 'prauc_sp': (0.692, 0.101), 'f1_sp': (0.666, 0.059),
        'auc_tp': 0.808, 'prauc_tp': 0.620, 'f1_tp': 0.521},
    'LR — after tuning': {
        'auc_sp': (0.835, 0.060), 'prauc_sp': (0.705, 0.102), 'f1_sp': (0.608, 0.118),
        'auc_tp': 0.807, 'prauc_tp': 0.618, 'f1_tp': 0.551},
    'RF — before tuning': {
        'auc_sp': (0.875, 0.022), 'prauc_sp': (0.769, 0.067), 'f1_sp': (0.709, 0.060),
        'auc_tp': 0.817, 'prauc_tp': 0.563, 'f1_tp': 0.549},
    'RF — after tuning': {
        'auc_sp': (0.877, 0.040), 'prauc_sp': (0.783, 0.066), 'f1_sp': (0.684, 0.072),
        'auc_tp': 0.816, 'prauc_tp': 0.558, 'f1_tp': 0.561},
}

def fmt(v):
    return f"{v[0]:.3f}±{v[1]:.3f}" if isinstance(v, tuple) else f"{v:.3f}"

print("="*104)
print("SPATIAL BLOCK CV (validation, mean ± std across 10 folds)")
print("="*104)
print(f"{'Model':22s} {'AUC-ROC':>16s} {'PR-AUC':>16s} {'F1':>16s}")
for k, r in results.items():
    print(f"{k:22s} {fmt(r['auc_sp']):>16s} {fmt(r['prauc_sp']):>16s} {fmt(r['f1_sp']):>16s}")

print("\n" + "="*104)
print("TEMPORAL HOLD-OUT (validation on unseen years ≥2020 — no leakage)")
print("="*104)
print(f"{'Model':22s} {'AUC-ROC':>10s} {'PR-AUC':>10s} {'F1':>10s}")
for k, r in results.items():
    print(f"{k:22s} {fmt(r['auc_tp']):>10s} {fmt(r['prauc_tp']):>10s} {fmt(r['f1_tp']):>10s}")

# Explicit deltas from tuning
print("\n" + "="*104)
print("EFFECT OF TUNING (Δ PR-AUC)")
print("="*104)
for model in ['LR', 'RF']:
    d_sp = results[f'{model} — after tuning']['prauc_sp'][0] - results[f'{model} — before tuning']['prauc_sp'][0]
    d_tp = results[f'{model} — after tuning']['prauc_tp']    - results[f'{model} — before tuning']['prauc_tp']
    print(f"  {model}: spatial {d_sp:+.3f}  |  temporal {d_tp:+.3f}")

SPATIAL BLOCK CV (validation, mean ± std across 10 folds)
Model                           AUC-ROC           PR-AUC               F1
LR — before tuning          0.836±0.029      0.692±0.101      0.666±0.059
LR — after tuning           0.835±0.060      0.705±0.102      0.608±0.118
RF — before tuning          0.875±0.022      0.769±0.067      0.709±0.060
RF — after tuning           0.877±0.040      0.783±0.066      0.684±0.072

TEMPORAL HOLD-OUT (validation on unseen years ≥2020 — no leakage)
Model                     AUC-ROC     PR-AUC         F1
LR — before tuning          0.808      0.620      0.521
LR — after tuning           0.807      0.618      0.551
RF — before tuning          0.817      0.563      0.549
RF — after tuning           0.816      0.558      0.561

EFFECT OF TUNING (Δ PR-AUC)
  LR: spatial +0.013  |  temporal -0.002
  RF: spatial +0.014  |  temporal -0.005
